In [ ]:
import cProfile
import pstats
import io

def profile_function(func):
    """Decorator to profile a function."""
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = 'cumulative'
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper


In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
import pandas as pd
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

In [ ]:
import numpy as np
from numba import njit

@njit
def ema_numba(close, length):
    out = np.full_like(close, np.nan)
    
    if len(close) <= length:
        return out
    
    alpha = 2 / (length + 1)
    ema = np.mean(close[:length])  # Initialize with SMA
    out[length-1] = ema
    
    for i in range(length, len(close)):
        ema = alpha * close[i] + (1 - alpha) * ema
        out[i] = ema
    
    return np.round(out, 2)

import numpy as np
from numba import njit

@njit
def jma_numba(src, length, phase, power):
    out = np.full_like(src, np.nan)
    
    if len(src) <= length:
        return out
    
    phaseRatio = 1.5 + phase / 100 if -100 <= phase <= 100 else (0.5 if phase < -100 else 2.5)
    beta = 0.45 * (length - 1) / (0.45 * (length - 1) + 2)
    alpha = np.power(beta, power)
    
    e0 = e1 = e2 = jma = 0.0
    
    for i in range(len(src)):
        if i == 0:
            e0 = src[i]
            e1 = 0.0
            e2 = 0.0
            jma = 0.0
        else:
            e0 = (1 - alpha) * src[i] + alpha * e0
            e1 = (src[i] - e0) * (1 - beta) + beta * e1
            e2 = (e0 + phaseRatio * e1 - jma) * np.power(1 - alpha, 2) + np.power(alpha, 2) * e2
            jma = e2 + jma
            
            # Reset values if they become too large or unstable
            if np.abs(jma) > 1e10:
                e0 = e1 = e2 = jma = 0.0
        
        if i >= length - 1:
            out[i] = np.round(jma, 2)
    
    return out

import numpy as np
from numba import njit

@njit
def alma_numba(series, length, offset, sigma):
    out = np.full_like(series, np.nan)
    
    if len(series) < length:
        return out
    
    for i in range(length - 1, len(series)):
        numerator = 0.0
        denominator = 0.0
        m = offset * (length - 1)
        s = length / sigma
        for j in range(length):
            weight = np.exp(-((j - m) ** 2) / (2 * s * s))
            numerator += weight * series[i - length + 1 + j]
            denominator += weight
        out[i] = round(numerator / denominator, 2)
    
    return out


@njit
def rsi_trail_numba(open, high, low, close, rsi_lower=45, rsi_upper=55, ma_length=20, ma_offset=0.85, ma_sigma=6):
    ohlc4 = (open + high + low + close) / 4
    ma = alma_numba(ohlc4, ma_length, ma_offset, ma_sigma)
    atr = atr_numba(high, low, close, 7)
    upper_bound = ma + (rsi_upper - 50) / 10 * atr
    lower_bound = ma - (50 - rsi_lower) / 10 * atr
    
    signal = np.zeros_like(close, dtype=np.float64)
    is_bullish = np.zeros_like(close, dtype=np.bool_)
    is_bearish = np.zeros_like(close, dtype=np.bool_)
    
    signal[:ma_length] = np.nan  # Set initial values to NaN
        
    for i in range(ma_length, len(close)):
        if ohlc4[i] > upper_bound[i]:
            if not is_bullish[i-1]:
                signal[i] = 1  # Bullish crossover
            is_bullish[i] = True
            is_bearish[i] = False
        elif close[i] < lower_bound[i]:
            if not is_bearish[i-1]:
                signal[i] = -1  # Bearish crossunder
            is_bullish[i] = False
            is_bearish[i] = True
        else:
            signal[i] = 0  # Neutral
            is_bullish[i] = is_bullish[i-1]
            is_bearish[i] = is_bearish[i-1]
    
    return signal, is_bullish, is_bearish

@njit
def atr_numba(high, low, close, length):
    true_range = np.zeros_like(close)
    atr = np.full_like(close, np.nan)
    
    for i in range(1, len(close)):
        true_range[i] = max(high[i] - low[i], abs(high[i] - close[i-1]), abs(low[i] - close[i-1]))
    
    for i in range(length, len(close)):
        if i == length:
            atr[i] = np.mean(true_range[:length])
        else:
            atr[i] = (atr[i - 1] * (length - 1) + true_range[i]) / length
    
    return atr


@njit
def chandelier_exit_numba(open, high, low, close, length=2, mult=1):
    atr = atr_numba(high, low, close, length) * mult
    
    long_stop = np.full_like(close, np.nan)
    short_stop = np.full_like(close, np.nan)
    direction = np.zeros_like(close)
    
    for i in range(length, len(close)):
        highest = np.max(close[i-length+1:i+1])
        lowest = np.min(close[i-length+1:i+1])
        
        long_stop[i] = highest - atr[i]
        short_stop[i] = lowest + atr[i]
        
        if i > length:
            long_stop[i] = max(long_stop[i], long_stop[i-1]) if close[i-1] > long_stop[i-1] else long_stop[i]
            short_stop[i] = min(short_stop[i], short_stop[i-1]) if close[i-1] < short_stop[i-1] else short_stop[i]
        
        if close[i] > short_stop[i-1]:
            direction[i] = 1
        elif close[i] < long_stop[i-1]:
            direction[i] = -1
        else:
            direction[i] = direction[i-1] if i > 0 else 0
    
    return long_stop, short_stop, direction

@njit()
def custom_round(x, decimals=2):
    """
    Custom rounding function for Numba (nopython mode).
    """
    multiplier = 10 ** decimals
    return np.floor(x * multiplier + 0.5) / multiplier

@njit()
def calculate_average_height_range(high, low, length):

    last_10_high = high[-length:]
    last_10_low = low[-length:]

    # Directly access the last element of high as a float
    current_candle_high = float(high[-1]) 
    current_candle_low = float(low[-1])

    # Calculate the height of each candle (high - low)
    avg_candle_heights = last_10_high - last_10_low

    average_height = np.mean(avg_candle_heights)

    current_candle_hight = current_candle_high - current_candle_low

    # Round to 2 decimal places
    average_height = custom_round(average_height)
    current_candle_hight = custom_round(current_candle_hight)
    
    return average_height, current_candle_hight
    

@njit
def calculate_current_bar_length(high, low):
    # Calculate the length of the current bar
    current_bar_length = high[-1] - low[-1]
    return current_bar_length

In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict, deque

feed_lock = threading.Lock()
range_length = 3
ema_length = 22
last_processed_candle = {}
trading_active = True
feed_opened = False
feedJson = defaultdict(lambda: deque(maxlen=10))  # Deque with maxlen=10 for each token
extra_feedJson = defaultdict(lambda: deque(maxlen=10))
resample_frequency = '5s'
resampling_lock = asyncio.Lock()
resampling_period_end = {}
last_resample_time = {}
resampled_data = {}
current_positions = {}
# Dictionary to keep track of entry instruments
entry_instruments = {}
entry_prices = {}
profit_target = 100
stop_loss = 20
bar_length_multiplier = 1.5
indicator_state = {}
finalized_data = {}
global signal_bar_high , signal_bar_low 
from concurrent.futures import ThreadPoolExecutor
trading_logic_task = None
option_symbols_subscribed = False

symbol = 'CRUDEOILM'
class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()
exchange = 'MCX'
Initial_token = '428870'
subscription_string = exchange + '|' + Initial_token
feed_lock = threading.Lock()


def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with feed_lock:
                if token == Initial_token:
                    # Append to the deque for the main token
                    feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                
                else:
                    # Append to the deque for extra tokens
                    extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    global finalized_data
    while True:
        if not feedJson:
            await asyncio.sleep(0.01)  # Sleep for a short interval if no data
            continue

        async with resampling_lock:
            for token, ticks in feedJson.items():
                if not ticks:
                    continue

                try:
                    # Create a DataFrame with the new ticks
                    df_new = pd.DataFrame(ticks)
                    df_new["tt"] = pd.to_datetime(df_new["tt"])
                    df_new.set_index("tt", inplace=True)

                    # Initialize or update the resampled DataFrame
                    if token not in resampled_data:
                        # Initialize the DataFrame with the first resample
                        resampled_data[token] = df_new['ltp'].resample(resample_frequency).ohlc()
                        last_resample_time[token] = df_new.index.max()
                    else:
                        # Resample the new ticks
                        df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                        # Update the existing resampled DataFrame with new data
                        for idx, row in df_resampled.iterrows():
                            if idx in resampled_data[token].index:
                                resampled_data[token].at[idx, 'high'] = max(resampled_data[token].at[idx, 'high'], row['high'])
                                resampled_data[token].at[idx, 'low'] = min(resampled_data[token].at[idx, 'low'], row['low'])
                                resampled_data[token].at[idx, 'close'] = row['close']
                            else:
                                resampled_data[token].loc[idx] = row
                        

                        # Update the last resampled time
                        last_resample_time[token] = df_new.index.max()
                    
                    

                    if token in resampled_data:
                        long_stop, short_stop, direction = chandelier_exit_numba(resampled_data[token]['open'].values,resampled_data[token]['high'].values,resampled_data[token]['low'].values,resampled_data[token]['close'].values)

                        resampled_data[token]['long_stop'] = long_stop
                        resampled_data[token]['short_stop'] = short_stop
                        resampled_data[token]['ce_direction'] = direction
                        
                    resampled_data[token] = resampled_data[token].dropna(subset=['open', 'high', 'low', 'close'])

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(0.01)  # Sleep for a short interval to avoid busy-waiting

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe([subscription_string])


def set_trading_active(value):
    global trading_active, trading_logic_task
    trading_active = value
    print(f"Trading active set to {trading_active}")

    if trading_active:
        if trading_logic_task is None or trading_logic_task.done():
            trading_logic_task = asyncio.create_task(trading_logic())
            print("Started trading logic task")
    else:
        if trading_logic_task is not None and not trading_logic_task.done():
            trading_logic_task.cancel()
            print("Cancelled trading logic task")


async def main():

    global trading_logic_task
   
    resample_task = asyncio.create_task(resample_ticks())
    connect_task = asyncio.create_task(connect_and_subscribe()) 
    trading_logic_task = asyncio.create_task(trading_logic())     
    update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_logic_task, update_symbols_task)

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()

# Always create a task for 'main' within the current loop
asyncio.create_task(main())

# If the loop wasn't running, start it
if not loop.is_running():
    loop.run_forever() 


Error processing tick data: 'ft'
Error processing tick data: 'ft'
[2024-08-02 22:53:35.815416] firing logic for: 428870
[2024-08-02 22:53:35.816762] 0.0, 0.0
[2024-08-02 22:53:35.818178] Token: 428870, Current Direction: 1.0, Previous Direction: 1.0
Error processing tick data: 'ft'
[2024-08-02 22:54:36.041097] firing logic for: 428870
[2024-08-02 22:54:36.041458] 1.0, 2.0
[2024-08-02 22:54:36.042089] Token: 428870, Current Direction: -1.0, Previous Direction: 1.0
[2024-08-02 22:54:36.042159] Error processing token 428870: '428870'
Error processing tick data: 'ft'
[2024-08-02 22:55:00.719486] firing logic for: 428870
[2024-08-02 22:55:00.719761] 0.67, 0.0
[2024-08-02 22:55:00.720380] Token: 428870, Current Direction: -1.0, Previous Direction: -1.0
[2024-08-02 22:55:06.119265] firing logic for: 428870
[2024-08-02 22:55:06.119502] 0.67, 0.0
[2024-08-02 22:55:06.120009] Token: 428870, Current Direction: -1.0, Previous Direction: -1.0
[2024-08-02 22:55:31.099977] firing logic for: 428870
[2

In [ ]:
async def trading_logic():
    
    global last_processed_candle, resampled_data, trading_active
    while trading_active:
        current_time = pd.Timestamp.now()        
        
        async with resampling_lock:
            for token, data in resampled_data.items():
                if not data.empty:
                    last_candle_start = data.index[-1]
                    resample_freq = data.index.freq or pd.Timedelta(seconds=5)
                    current_candle_end = last_candle_start + resample_freq

                    # Check if the current time is beyond the current candle's end time
                    if current_time >= current_candle_end:
                        # Ensure we haven't processed this candle yet
                        
                        if token not in last_processed_candle or last_processed_candle[token] < current_candle_end:
                            last_processed_candle[token] = current_candle_end
                            
                            # Add the logic to process the resampled data here
                            print(f"[{pd.Timestamp.now()}] firing logic for: {token}")
                            await process_token_data(token)

            # Sleep briefly to yield control and avoid busy-waiting
            await asyncio.sleep(.5)  # Adjust the sleep duration as needed


In [ ]:
async def process_token_data(token):
    """
    Processes resampled candle data for a given token, calculates indicators, and executes trade logic.

    Args:
        token: The token or symbol to process.
    """
    df = resampled_data.get(token)

    if df is None or len(df) < 3:
        print(f"[{pd.Timestamp.now()}] Not enough data or token {token} not found. Need at least 3 candles.")
        return

    try:
        # Convert relevant columns to NumPy arrays
        high = df['high'].values
        low = df['low'].values

        # Calculate indicators (assuming you have the 'calculate_average_height_range' function)
        average_height, current_candle_height = calculate_average_height_range(high, low, range_length)
        print(f"[{pd.Timestamp.now()}] {average_height}, {current_candle_height}")

        # Get the current time
        current_time = pd.Timestamp.now()

        # Get the end of the last completed candle 
        new_candle_start_time = current_time.floor(resample_frequency)

        # Calculate the start of the last completed candle
        resample_freq_timedelta = pd.Timedelta(resample_frequency)
        last_completed_candle_start_time = new_candle_start_time - resample_freq_timedelta

        # Filter completed candles
        if new_candle_start_time in df.index:
            completed_candles_df = df.loc[:new_candle_start_time - pd.Timedelta(microseconds=1)] 
        elif last_completed_candle_start_time in df.index:
            completed_candles_df = df

        try:
            # Check if expected columns exist
            required_columns = ['ce_direction'] 
            missing_columns = [col for col in required_columns if col not in completed_candles_df.columns]
            if missing_columns:
                raise KeyError(f"Missing columns {missing_columns}")

            # Extract current and previous directions
            current_direction = completed_candles_df['ce_direction'].iloc[-1]
            previous_direction = completed_candles_df['ce_direction'].iloc[-2]

            print(f"[{pd.Timestamp.now()}] Token: {token}, Current Direction: {current_direction}, Previous Direction: {previous_direction}")

            # Execute trade logic
            await execute_trade_logic(token, current_direction, previous_direction, completed_candles_df) 

        except KeyError as e:
            print(f"[{pd.Timestamp.now()}] Error processing token {token}: {e}") 

    except KeyError as e:
        print(f"[{pd.Timestamp.now()}] Error processing token {token}: Column {e} not found.") 
    except IndexError:
        print(f"[{pd.Timestamp.now()}] Error processing token {token}: Not enough candles to calculate indicators.")


In [ ]:
import time
from threading import Lock

# Global lock for synchronization
trade_execution_lock = Lock()

async def execute_trade_logic(token, current_direction, previous_direction, completed_candles_df):
    global current_positions 

    with trade_execution_lock: 

        # Exit Logic (Prioritized)
        if token in current_positions: 
            position_type_to_exit = next((key for key in current_positions[token] if current_positions[token][key]), None)
            
            if position_type_to_exit:
                op_token = current_positions[token][position_type_to_exit]["option_token"]
                op_symbol = current_positions[token][position_type_to_exit]["symbol_name"]

                if (position_type_to_exit == 'call_buy' and current_direction == -1 and previous_direction == 1) or \
                   (position_type_to_exit == 'put_buy' and current_direction == 1 and previous_direction == -1):
                    
                    price = get_latest_price_option(op_token)
                    if price is not None:
                        action_time = pd.Timestamp.now()
                        #log_trade(timestamp, op_symbol, "EXIT", price, action_time) 
                        print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Exited {position_type_to_exit} trade for {op_symbol} at {action_time}")
                        del current_positions[token][position_type_to_exit]

        # Entry Logic 
        if token not in current_positions or not current_positions[token]:  
            if current_direction == 1 and previous_direction == -1: 
                #option_token, symbol_name = enter_trade(token, "call_buy", state.ce_trading_symbol)
                current_positions[token]["call_buy"] = {
                    "option_token": state.ce_trading_token,
                    "symbol_name": state.ce_trading_symbol
                }
                print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Entered call buy trade for {token}")
            elif current_direction == -1 and previous_direction == 1:
                #option_token, symbol_name = enter_trade(token, "put_buy", state.pe_trading_symbol)
                current_positions[token]["put_buy"] = {
                    "option_token": state.pe_trading_token,
                    "symbol_name": state.pe_trading_symbol
                }
                print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Entered put buy trade for {token}")


In [26]:
current_positions

{}

In [ ]:
def get_latest_price_option(entry_instrument):
    entry_instrument_str = str(entry_instrument)
    with resampling_lock:
        if entry_instrument_str in extra_feedJson:
            latest_data = extra_feedJson[entry_instrument_str][-1]
            latest_price = latest_data['ltp']
            
            return latest_price
        else:
            print(f"{entry_instrument_str} not found in extra_feedJson")  # Debug print
            return None
        
def get_latest_price_index(entry_instrument):
    entry_instrument_str = str(entry_instrument)
    with resampling_lock:
        if entry_instrument_str in extra_feedJson:
            latest_data = extra_feedJson[entry_instrument_str][-1]
            latest_price = latest_data['ltp']
            
            return latest_price
        else:
            print(f"{entry_instrument_str} not found in extra_feedJson")  # Debug print
            return None

In [ ]:

def find_atm_strike_and_symbols():   

    global  atm_strike
    cmp_bn = float(resampled_data[Initial_token]['close'].iloc[-1]) 

    # Check if cmp_bn is NaN
    if pd.isna(cmp_bn):
        #print(f"NaN value encountered for {Initial_token} close price. Skipping ATM calculation.")
        return None, None, None, None 
    cmp_bn = int(float(cmp_bn))
    mod = cmp_bn % 100
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)
    #print(atm_strike)

    if exchange == 'NSE':
        # Assuming fno_scrips_mcx is a global DataFrame
        filtered_df = fno_scrips[
            (fno_scrips['Symbol'] == symbol) &
            (fno_scrips['StrikePrice'] == float(atm_strike))
        ]
    elif exchange == 'MCX':
        filtered_df = fno_scrips_mcx[
            (fno_scrips_mcx['Symbol'] == symbol) &
            (fno_scrips_mcx['StrikePrice'] == float(atm_strike))
        ]

    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token


async def update_atm_symbols():
    global option_symbols_subscribed, trade_execution_lock # Assuming trade_execution_lock is defined globally
    while True:         
        try:
            result = await asyncio.to_thread(find_atm_strike_and_symbols)

            if result[0] is not None:
                # Acquire the lock before updating state variables
                with trade_execution_lock:
                    (state.ce_trading_symbol, state.pe_trading_symbol,
                     state.ce_trading_token, state.pe_trading_token) = result

                if not option_symbols_subscribed:
                    option_subscription()
                    option_symbols_subscribed = True
            else:
                print("Failed to update symbols")

        except Exception as e:
            print(f"Error in update_atm_symbols: {e}")        
        await asyncio.sleep(15)


In [ ]:
def option_subscription():
    op_chain = api.get_option_chain(exchange='MCX', tradingsymbol=state.ce_trading_symbol, strikeprice=atm_strike, count =2)
    tokens = [item['token'] for item in op_chain['values']]
    subscriptions = [f"MCX|{token}" for token in tokens]
    # Subscribe to each token
    for sub in subscriptions:
        api.subscribe(sub)

    # Print the subscriptions for verification
    print("Subscribed to:", subscriptions)

In [25]:
state.ce_trading_token

434900